In [1]:
#Libs
from delta.tables import DeltaTable
from pyspark.sql import functions as F

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 5, Finished, Available, Finished, False)

# Great exp notebook utils

In [8]:
%run ./gex_demo_utils_nb

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 47, Finished, Available, Finished, True)

# Create rules

In [3]:
spark.sql("""
INSERT OVERWRITE dq_rules VALUES
('orders','R0','Schema','schema','*','expect_schema',
 '{"columns":[
     {"name":"order_line_id","type":"IntegerType"},
     {"name":"order_id","type":"IntegerType"},
     {"name":"customer_id","type":"IntegerType"},
     {"name":"product_id","type":"IntegerType"},
     {"name":"quantity","type":"IntegerType"}, 
     {"name":"unit_price","type":"DoubleType"},
     {"name":"order_total","type":"DoubleType"},
     {"name":"order_date","type":"DateType"},
     {"name":"email","type":"StringType"},
     {"name":"country_code","type":"StringType"}
 ]}',
 'error', true),

('orders','R1','Completeness','not_null','order_date','expect_not_null','', 'error',true),
('orders','R2','Uniqueness','unique','order_line_id','expect_unique','', 'error',true),
('orders','R3','Validity','regex','email','expect_regex','^[A-Za-z0-9+_.-]+@(.+)$','error',true),
('orders','R4','Validity','reference','country_code','expect_reference','countries_reference','error',true),
('orders','R5','Linkage','fk','customer_id','expect_fk','customers_reference','error',true),
('orders','R6','Linkage','fk','product_id','expect_fk','products_reference','error',true),
('orders','R7','Accuracy','custom_sql','order_total','expect_custom_sql','SELECT * FROM {batch} WHERE order_total != quantity * unit_price','error',true),

('orders','R8','Schema','column_type','*','expect_column_types',
 '{"columns":[
     {"name":"order_line_id","type":"IntegerType"},
     {"name":"order_id","type":"IntegerType"},
     {"name":"customer_id","type":"IntegerType"},
     {"name":"product_id","type":"IntegerType"},
     {"name":"quantity","type":"IntegerType"},
     {"name":"unit_price","type":"DoubleType"},
     {"name":"order_total","type":"DoubleType"},
     {"name":"order_date","type":"DateType"},
     {"name":"email","type":"StringType"},
     {"name":"country_code","type":"StringType"}
 ]}', 'error', false)
""")

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 24, Finished, Available, Finished, False)

DataFrame[]

In [4]:
%%sql 
SELECT * FROM dq_rules WHERE dataset_name='orders'
order by rule_id;

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 25, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 9 fields>

In [5]:
%%sql
SELECT *
FROM silver_orders
ORDER BY order_line_id

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 10 fields>

# Read data from bronze files with wrong schema

In [6]:
ds = {
    "dataset_name": "orders",
    "bronze_path": "Files/bronze/orders_raw_bad_schema",
    "silver_table": "dq_lh.dbo.silver_orders",
    "pk_column": "order_line_id"
}

df = spark.read.parquet(ds["bronze_path"])

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 27, Finished, Available, Finished, False)

In [9]:
df, success, validation_results = run_schema_validation_gx(df, ds['dataset_name'])

run_id = log_schema_result(ds['dataset_name'], df, success, validation_results=validation_results)

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 48, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]


Schema Validation Summary for dataset 'orders':
Rule: expect_table_columns_to_match_ordered_list | Column: TABLE | Success: False | Details: {'observed_value': ['order_line_id', 'order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'order_total', 'order_date', 'email'], 'details': {'mismatched': [{'Expected Column Position': 9, 'Expected': 'country_code', 'Found': None}]}}

Schema validation logged with run_id: da4bcce9-914a-4a62-9b1b-aa697bf1f1d0


Exception: Schema validation FAILED for dataset 'orders'. Run ID: da4bcce9-914a-4a62-9b1b-aa697bf1f1d0. See dq_schema_log for details.

# Read data from bronze files with correct schema

In [10]:
ds = {
    "dataset_name": "orders",
    "bronze_path": "Files/bronze/orders_raw",
    "silver_table": "dq_lh.dbo.silver_orders",
    "pk_column": "order_line_id"
}

df = spark.read.parquet(ds["bronze_path"])

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 49, Finished, Available, Finished, False)

In [11]:
df, success, validation_results = run_schema_validation_gx(df, "orders")

run_id = log_schema_result("orders", df, success, validation_results=validation_results)

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 50, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]


Schema Validation Summary for dataset 'orders':
Rule: expect_table_columns_to_match_ordered_list | Column: TABLE | Success: True | Details: {'observed_value': ['order_line_id', 'order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'order_total', 'order_date', 'email', 'country_code']}

Schema validation logged with run_id: d9a172b8-96cb-4036-88a2-1bfd9100f6cf


StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 51, Finished, Available, Finished, False)

# Schema log

In [12]:
display( spark.sql("SELECT * FROM dq_lh.dbo.dq_schema_log order by run_time desc limit 2") )

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00f78a1c-8819-4cf3-b147-859902ca4098)

# 

# Validate DQ rules

In [13]:
passed_df, failed_df = run_gx_validation(df, ds["dataset_name"], ds["pk_column"])

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 53, Finished, Available, Finished, False)


=== Running Custom SQL Rule: R7 ===
Column: order_total
SQL being executed:
SELECT * FROM {batch} WHERE order_total != quantity * unit_price



Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Custom SQL rule R7 detected 1 failing rows.
Sample failing row: {'order_line_id': 1000, 'order_id': 1, 'customer_id': 100, 'product_id': 1, 'quantity': 1, 'unit_price': 100.0, 'order_total': 999.0, 'order_date': datetime.date(2026, 3, 10), 'email': 'a@email.com', 'country_code': 'US'}


Calculating Metrics:   0%|          | 0/14 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]


Data Quality Validation Summary for dataset 'orders':
Total rows: 53
Passed rows: 20
Failed rows: 33

Rule Results:
Rule: unexpected_rows_expectation | Column: N/A | Success: False | Failed rows: 1
Rule: expect_column_values_to_be_in_set | Column: country_code | Success: False | Failed rows: 13
Rule: expect_column_values_to_match_regex | Column: email | Success: False | Failed rows: 1
Rule: expect_column_values_to_be_in_set | Column: customer_id | Success: False | Failed rows: 16
Rule: expect_column_values_to_be_in_set | Column: product_id | Success: False | Failed rows: 8
Rule: expect_column_values_to_not_be_null | Column: order_date | Success: False | Failed rows: 1
Rule: expect_column_values_to_be_unique | Column: order_line_id | Success: False | Failed rows: 2


StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 54, Finished, Available, Finished, False)

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 55, Finished, Available, Finished, False)

In [14]:
write_failed_and_log(df, passed_df, failed_df, ds["dataset_name"], ds["pk_column"])

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 56, Finished, Available, Finished, False)

DQ run logged successfully: 5561b473-30cf-423c-9d58-191a2c5edd5f


In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_orders (
    order_line_id INT,
    order_id INT,
    customer_id INT,
    product_id INT,
    quantity INT,
    unit_price DOUBLE,
    order_total DOUBLE,
    order_date DATE,
    email STRING,
    country_code STRING
)
""")

In [15]:
from delta.tables import DeltaTable

silver = DeltaTable.forName(spark, "dq_lh.dbo.silver_orders")

silver.alias("tgt").merge(
    passed_df.alias("src"),
    "tgt.order_line_id = src.order_line_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 57, Finished, Available, Finished, False)

In [17]:
display(failed_df)

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d351f104-0108-457d-ba27-adce4ada4974)

In [16]:
%%sql
SELECT * 
FROM silver_orders
ORDER BY order_line_id

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 58, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 10 fields>

In [18]:
%%sql
SELECT *
FROM dq_failed_rows

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 60, Finished, Available, Finished, False)

<Spark SQL result set with 199 rows and 4 fields>

In [19]:
%%sql
SELECT *
FROM dq_run_log

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 61, Finished, Available, Finished, False)

<Spark SQL result set with 27 rows and 6 fields>

In [ ]:
# Example: apply filters, derived columns, or other transformations before DQ
# df = df.filter(df["order_total"] > 0)
# df = df.withColumn("processed_ts", F.current_timestamp())

StatementMeta(, a1d6d9cd-779d-4c07-b480-965fb28f8ca6, -1, Cancelled, , Cancelled, True)

In [20]:
%%sql
SELECT
    pk,
    run_id,
    dataset_name,
    CAST(get_json_object(record_data, '$.order_line_id') AS INT) AS order_line_id,
    CAST(get_json_object(record_data, '$.order_id') AS INT) AS order_id,
    CAST(get_json_object(record_data, '$.customer_id') AS INT) AS customer_id,
    CAST(get_json_object(record_data, '$.product_id') AS INT) AS product_id,
    CAST(get_json_object(record_data, '$.quantity') AS INT) AS quantity,
    CAST(get_json_object(record_data, '$.unit_price') AS DOUBLE) AS unit_price,
    CAST(get_json_object(record_data, '$.order_total') AS DOUBLE) AS order_total,
    CAST(get_json_object(record_data, '$.order_date') AS DATE) AS order_date,
    get_json_object(record_data, '$.email') AS email,
    get_json_object(record_data, '$.country_code') AS country_code
FROM dq_failed_rows
WHERE dataset_name = 'orders'
ORDER BY order_line_id;

StatementMeta(, 3b8c2405-028a-4f1c-9edf-5c13041225a3, 62, Finished, Available, Finished, False)

<Spark SQL result set with 199 rows and 13 fields>